# Proyecto 1 — Sacarle al EEG el ruido de los ojos (EOG) y de los músculos (EMG) con NLMS

**Cómo leer este notebook:** esta primera parte explica el problema y la idea, sin código. Después, cada sección empieza con un recuadro **"Dónde estamos"**, que dice qué pregunta responde esa sección. Debajo de cada gráfico hay un **"Qué mirar"** que explica por qué se dibujó y qué conclusión sale.

---

## 1. El problema

### 1.1 Qué mide un electrodo
Un electrodo pegado en la cabeza no mide solo al cerebro. Mide la **suma** de tres cosas:

- **`s(n)`, el EEG:** la actividad del cerebro. Es lo que queremos. Es muy chico: decenas de µV.
- **`η_EOG(n)`, el ruido de los ojos:** el ojo funciona como una pila (la córnea es positiva respecto de la retina). Cuando los ojos se mueven o parpadean, ese campo eléctrico cambia y llega a los electrodos de la frente. Es grande (hasta cientos de µV) y **lento**: casi todo está por debajo de ~15 Hz.
- **`η_EMG(n)`, el ruido de los músculos:** cuando se contrae un músculo (mandíbula, sien, cuello), genera una señal eléctrica **rápida** (más de 20 Hz). Aparece en **ráfagas**: nada durante un rato y de golpe una explosión.

Entonces lo que se graba es

`y(n) = s(n) + λ₁·η_EOG(n) + λ₂·η_EMG(n)`

donde `λ₁` y `λ₂` dicen cuánto de cada ruido llega a *ese* electrodo: cuanto más cerca de la fuente, más grande. **El objetivo es recuperar `s(n)` a partir de `y(n)`.**

### 1.2 Por qué no alcanza con un filtro común
Si el ruido estuviera en frecuencias donde no hay EEG, un pasa-banda lo sacaría y listo. El problema es que **se pisan**:
- el EOG (< 15 Hz) cae encima de las bandas δ, θ y α del EEG;
- el EMG (> 20 Hz) cae encima de β y γ.

Un filtro que saque el EOG también se lleva el EEG de esas frecuencias.

### 1.3 La idea: cancelación adaptativa de ruido
Funciona como los auriculares con cancelación de ruido: un micrófono de afuera escucha *solo el ruido*, el auricular arma una copia del ruido tal como llega al oído, y la resta.

Acá es igual. Si tenemos un sensor que mide (casi) solo el ruido, al que llamamos **referencia** `x(n)`:
1. pasamos la referencia por un filtro `w` que la convierte en "el ruido tal como llega al electrodo". A esa estimación la llamamos `ŷ(n)`;
2. la restamos: `e(n) = y(n) − ŷ(n)`.

Si `ŷ` es igual al ruido que hay en `y`, lo que queda es `e(n) = s(n)`: **el EEG limpio**.

**¿Cómo encuentra el filtro el `w` correcto?** Va ajustando `w` para que `e(n)` sea lo más chico posible.

**¿Y eso no borra también el EEG?** No, mientras la referencia no tenga EEG. El filtro solo puede armar cosas a partir de la referencia. Si en la referencia no hay EEG, no puede armar EEG para restarlo. Lo único que puede achicar es la parte de ruido, y el EEG queda en `e(n)` (principio de ortogonalidad, `6. LMS.pdf`).

**El riesgo:** si la referencia sí trae algo de EEG, el filtro puede restar EEG bueno. Esto aparece más adelante.

### 1.4 El diagrama en bloques

```
            s(n) EEG ─────────────┐
 η_EOG ──λ₁───────────────────────┤
 η_EMG ──λ₂───────────────────────┼──►  y(n) = s + λ₁η_EOG + λ₂η_EMG   (electrodo)
                                  │              │
 x₁(n) (canal EOG) ──► [ w₁ ] ──┐ │              ▼
                                ├─(+)── ŷ(n) ──►(−)──► e(n) = y − ŷ ≈ s(n)
 x₂(n) (ref. EMG)  ──► [ w₂ ] ──┘                 │
        ▲                                         │
        └──────── NLMS: w = [w₁; w₂] ◄────────────┘
```

### 1.5 Qué es cada señal en este dataset

| Símbolo | Qué es acá |
|---|---|
| `y(n)` | Un canal de EEG tal como se grabó (Fp1, F7, F8, T8 u Oz). |
| `x₁(n)` | El canal **EOG** del dataset: un electrodo cerca del ojo. |
| `x₂(n)` | El dataset **no tiene canal de EMG**, así que lo armamos con los electrodos más cercanos a la mandíbula y la sien: `T9, TP9, FT9` (izquierda) y `T10, TP10, FT10` (derecha). Promediamos los tres de cada lado y les aplicamos un pasa-altos en 25 Hz: el EMG está arriba de 20 Hz, y así sacamos el EEG lento que también miden esos electrodos. Quedan **dos referencias de EMG** (izquierda y derecha), o sea 3 referencias en total. |
| `w` | Tres filtros de M = 3 coeficientes, uno por referencia, juntos en un solo vector de 9 números. |
| NLMS | La regla que ajusta `w` en cada muestra. |
| `e(n)` | La salida: nuestra estimación del EEG limpio. |

**Importante:** hay **una sola** señal de error, `e(n)`, y ajusta todos los coeficientes a la vez. EOG y EMG no son "dos errores": son dos *ruidos*, cada uno con su referencia.

---

## 2. El problema de fondo: ¿cómo sabemos si salió bien?
Para saber si `e(n)` es el EEG limpio, lo natural sería compararlo con `s(n)`. Pero **en una grabación real nadie conoce `s(n)`**: solo tenemos `y(n)`. Eso es lo que significa "no hay ground truth". Por eso el notebook va por dos caminos.

**Camino A, experimento semi-sintético (§4).** Armamos la mezcla nosotros:
- tomamos un pedazo de EEG que ya está limpio y lo llamamos `s`;
- le sumamos ruido real de *otra* grabación, en la cantidad que elegimos;
- se lo damos al filtro.

Como `s` lo elegimos nosotros, lo conocemos, y podemos medir exactamente cuánto se parece `e` a `s`. De acá salen los números confiables (curva de aprendizaje, desajuste, error relativo…). La limitación es que la mezcla armada puede ser más fácil que la real.

**Camino B, datos reales (§5 y §6).** Corremos el filtro sobre la grabación cruda. No podemos comparar `e` con `s`. En cambio, nos preguntamos **"si el filtro anduviera bien, ¿qué tendría que pasar?"** y lo comprobamos. Por ejemplo:
- el ruido de los ojos tendría que desaparecer de `e`;
- el ritmo α del cerebro no tendría que cambiar;
- un canal lejos de ojos y músculos no tendría que cambiar.

Ninguna prueba sola alcanza, pero si todas coinciden, le podemos creer.

---

## 3. Mapa del notebook

| Sección | Pregunta que responde | Camino |
|---|---|---|
| §1 Carga | ¿Cómo leo los datos? | — |
| §2 Inspección | ¿Qué ruido hay *realmente* en este sujeto? (decide todo lo demás) | — |
| §3 El filtro | El código de NLMS y de Wiener | — |
| §4 Semi-sintético | ¿El filtro funciona? ¿Con qué μ̃ y M? | A |
| §5 Datos reales | ¿Qué hace el filtro sobre la grabación cruda? | B |
| §6 Validación | ¿Le creemos a `e(n)` en los datos reales? | B |
| §7 Conclusiones | ¿Obtuvimos un EEG limpio? | — |

**Resumen de resultados** (el detalle está en cada sección):
- El filtro está bien programado y, en el caso armado, limpia: con μ̃ = 0.03 y M = 3 el error relativo baja de 2.0 a 0.67.
- Este dataset viene **preprocesado**: ya le sacaron los parpadeos al EEG. Quedan movimientos de ojos hacia los costados y algunas ráfagas de músculo.
- En los datos reales **todavía no podemos afirmar** que `e` sea EEG limpio:
  - las ráfagas de músculo se van bien;
  - pero el filtro también saca alta frecuencia donde no hay músculo;
  - y a veces resta *de más* el ruido de los ojos.

In [ ]:
import glob
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, lfilter, welch, find_peaks

mne.set_log_level("ERROR")

# Parámetros
RAIZ = "./datasets-presentacion-PASA/ds008701"
SUJETO = "01"
FS = 250                    # Submuestreo 1000 -> 250 Hz: la banda grabada termina en ~80 Hz
T_INI, T_FIN = 10, 110      # Ventana útil [s]: los bordes tienen derivas de varios mV
CANALES_EMG_IZQ = ["T9", "TP9", "FT9"]
CANALES_EMG_DER = ["T10", "TP10", "FT10"]
CANALES_REF = ["EOG"] + CANALES_EMG_IZQ + CANALES_EMG_DER
F_HP_EMG = 25               # Pasa-altos de la referencia de EMG [Hz]: saca δ–α del EEG

## §1. Carga de los datos

> **Dónde estamos:** todavía no filtramos nada. Solo leemos las grabaciones y las dejamos listas.

**Qué hay en el disco:**
- El sujeto 01 escuchó 20 canciones. Cada carpeta `ses-tXX` es **una canción**, no una sesión en el tiempo.
- Cada grabación tiene 66 canales de EEG y 1 de EOG, a 1000 muestras por segundo, en voltios.

**Qué hace el código:**
- Lee los archivos `.set` con MNE. Qué canal es EOG se lee de `channels.tsv`, porque MNE marca todos como EEG.
- Pasa todo a µV.
- **Recorta a 10–110 s.** Los primeros y últimos segundos tienen derivas enormes (de mV) que no son ni EEG ni ruido útil. Si se las damos al filtro, se dedica a seguir eso.
- **Baja a 250 muestras por segundo.** La grabación ya viene filtrada y no tiene nada arriba de ~80 Hz (lo vas a ver en la PSD de §2.2). Con 250 muestras/s alcanza (Nyquist = 125 Hz > 80 Hz), y el filtro hace 4 veces menos cuentas.
- Saca la continua (el promedio) de cada canal.

**Resultado:** `DATOS[canción][canal]` es un arreglo de 25 001 muestras (100 s) en µV.

In [ ]:
def ruta_set(ses):
    return f"{RAIZ}/sub-{SUJETO}/ses-{ses}/eeg/sub-{SUJETO}_ses-{ses}_task-music_eeg.set"


def cargar(ses, fs=FS):
    """Carga una canción: dict {canal: señal en µV}, recortada a [T_INI, T_FIN] y submuestreada a fs."""
    raw = mne.io.read_raw_eeglab(ruta_set(ses), preload=True)
    raw.crop(T_INI, T_FIN)
    raw.resample(fs)                        # MNE aplica el antialias
    X = raw.get_data() * 1e6                # V -> µV
    X = X - X.mean(axis=1, keepdims=True)   # Sin continua
    return dict(zip(raw.ch_names, X))


SESIONES = sorted(p.split("/")[-3][4:] for p in glob.glob(f"{RAIZ}/sub-{SUJETO}/ses-*/eeg/*_eeg.set"))

base_ses = f"{RAIZ}/sub-{SUJETO}/ses-t01/eeg/sub-{SUJETO}_ses-t01"
canales_tsv = pd.read_csv(f"{base_ses}_task-music_channels.tsv", sep="\t", encoding="utf-8-sig")
electrodos = pd.read_csv(f"{base_ses}_space-CapTrak_electrodes.tsv", sep="\t", encoding="utf-8-sig", na_values="n/a")
tipos = dict(zip(canales_tsv["name"], canales_tsv["type"]))
CANALES_EEG = [c for c, t in tipos.items() if t == "EEG"]

DATOS = {ses: cargar(ses) for ses in SESIONES}

print(f"{len(SESIONES)} canciones: {SESIONES}")
print(f"{len(CANALES_EEG)} canales EEG + {sum(t == 'EOG' for t in tipos.values())} EOG, "
      f"{len(DATOS[SESIONES[0]]['Oz']) / FS:.1f} s por canción a {FS} Hz")

## §2. Inspección: ¿qué ruido hay realmente?

> **Dónde estamos:** antes de diseñar el filtro hay que saber contra qué peleamos. Si el EEG no tiene ruido de ojos, no hay nada que cancelar. Si los canales temporales no tienen EMG, no sirven como referencia.

**Las preguntas, y con qué herramienta se responde cada una:**

1. **¿El ruido de los ojos aparece en el EEG?** Se mide la **correlación** entre el canal EOG y cada canal de EEG.
   - La correlación es un número entre −1 y 1 que dice cuánto se mueven juntas dos señales.
   - ≈ 0 es que no tienen nada que ver; cerca de 1, que suben y bajan juntas; cerca de −1, que una sube cuando la otra baja.
   - El filtro **solo puede sacar el EOG de los canales que se parecen al EOG**.
2. **¿Hay parpadeos?** Un parpadeo es un pico grande y corto en el EOG. Los contamos con un umbral. El código los llama "parpadeos", pero en realidad cuenta cualquier evento grande del EOG, incluidos los movimientos de ojos.
3. **¿Hay EMG en los canales temporales?** El EMG viene en ráfagas. Para detectarlo:
   - nos quedamos con lo que está arriba de 30 Hz, que es donde está el EMG;
   - calculamos la **envolvente**: el "tamaño" de la señal a lo largo del tiempo;
   - medimos su **curtosis**, un número que dice qué tan "picuda" es una señal. Un ruido parejo da ≈ 3; una señal tranquila con algunas explosiones da mucho más que 3;
   - miramos si las ráfagas pasan a la vez a la izquierda y a la derecha (correlación entre las dos envolventes).

En esta celda también se arman las referencias de EMG (`EMG_IZQ`, `EMG_DER`) que se usan en el resto del notebook.

In [ ]:
sos_lp10 = butter(4, 10, "low", fs=FS, output="sos")
sos_hp30 = butter(4, 30, "high", fs=FS, output="sos")
sos_lp2 = butter(4, 2, "low", fs=FS, output="sos")
sos_hp_emg = butter(4, F_HP_EMG, "high", fs=FS, output="sos")


def envolvente_hf(x):
    """Envolvente de la parte > 30 Hz (indicador de EMG)."""
    return sosfiltfilt(sos_lp2, np.abs(sosfiltfilt(sos_hp30, x)))


def curtosis(x):
    x = x - x.mean()
    return np.mean(x**4) / np.mean(x**2)**2


def detectar_parpadeos(eog):
    """Picos del EOG pasa-bajos que superan 4 desvíos robustos (MAD)."""
    eog_lp = sosfiltfilt(sos_lp10, eog)
    umbral = 4 * np.median(np.abs(eog_lp)) / 0.6745
    picos, _ = find_peaks(np.abs(eog_lp), height=umbral, distance=int(0.3 * FS))
    return picos, eog_lp


def ref_emg(c, canales):
    """Referencia de EMG: promedio de los canales de un lado, pasa-altos en F_HP_EMG."""
    return sosfiltfilt(sos_hp_emg, np.mean([c[n] for n in canales], axis=0))


EMG_IZQ = {ses: ref_emg(c, CANALES_EMG_IZQ) for ses, c in DATOS.items()}
EMG_DER = {ses: ref_emg(c, CANALES_EMG_DER) for ses, c in DATOS.items()}

filas = []
for ses, c in DATOS.items():
    picos, _ = detectar_parpadeos(c["EOG"])
    corr = {n: np.corrcoef(c["EOG"], c[n])[0, 1] for n in CANALES_EEG}
    n_max = max(corr, key=lambda n: abs(corr[n]))
    env_izq, env_der = envolvente_hf(EMG_IZQ[ses]), envolvente_hf(EMG_DER[ses])
    filas.append(dict(sesion=ses, parpadeos=len(picos), std_EOG=c["EOG"].std(), std_Fp1=c["Fp1"].std(),
                      corr_EOG_Fp1=corr["Fp1"], corr_EOG_max=corr[n_max], canal_max=n_max,
                      kurt_emg_izq=curtosis(env_izq), kurt_emg_der=curtosis(env_der),
                      corr_env_izq_der=np.corrcoef(env_izq, env_der)[0, 1]))

tabla = pd.DataFrame(filas).set_index("sesion")
print(tabla.round(2).to_string())

### Qué mirar en la tabla

Cada fila es una canción. Las columnas:
- `parpadeos`: cantidad de eventos grandes del EOG.
- `std_EOG`, `std_Fp1`: tamaño típico (desvío estándar) en µV. El EOG es unas 5 veces más grande que el EEG.
- `corr_EOG_Fp1`: correlación del EOG con Fp1, el electrodo justo arriba del ojo izquierdo.
- `corr_EOG_max`, `canal_max`: el canal de EEG que más se parece al EOG, y cuánto.
- `kurt_emg_izq`, `kurt_emg_der`: curtosis de la envolvente de cada referencia de EMG (mucho más que 3 = hay ráfagas).
- `corr_env_izq_der`: si las ráfagas pasan a la vez de los dos lados.

**Qué sale:**
- El canal que más se parece al EOG casi siempre es **F7, con signo +, o F8, con signo −**. F7 está al costado del ojo izquierdo y F8 al costado del derecho.
- Que tengan signos opuestos es justo lo que pasa cuando los ojos **miran hacia un costado**: un lado se hace positivo y el otro negativo. Conclusión: **en el EEG quedó ruido de movimientos oculares horizontales**. La canción con más es **t17** (r ≈ 0.6).
- Correlaciones de 0.2 a 0.6 no son enormes: el ruido de ojos está, pero es una parte del EEG, no la mayor parte.
- La curtosis del EMG es mucho más que 3 **solo en t04, t07, t15 y t25**: ahí hay ráfagas de músculo. La que más tiene es **t15**. En las demás canciones, las referencias de EMG son ruido parejo.
- Las ráfagas pasan a la vez de los dos lados (correlación de hasta 1.0). Es algo que mueve los dos lados juntos: apretar la mandíbula o tragar.

In [ ]:
# ============================================================
# 2.1 EOG vs frontales: ¿queda EOG en el EEG?
# ============================================================

SES_EOG = tabla["corr_EOG_max"].abs().idxmax()      # Canción donde el EOG más se ve en el EEG
c = DATOS[SES_EOG]
t = np.arange(len(c["EOG"])) / FS + T_INI

fig, axes = plt.subplots(5, 1, figsize=(14, 9), sharex=True)
for ax, n in zip(axes, ["EOG", "F7", "F8", "Fp1", "Oz"]):
    ax.plot(t, c[n], linewidth=0.5)
    ax.set_ylabel(f"{n} [µV]")
    ax.grid(True)
axes[0].set_title(f"sub-{SUJETO} ses-{SES_EOG}")
axes[-1].set_xlabel("t [s]")
plt.tight_layout()
plt.show()

### Qué mirar: el ruido de los ojos a simple vista (t17)

**Por qué está:** para ver con los ojos lo que la tabla dijo con números, en la canción con más ruido de ojos.
- **Arriba, el EOG:** subidas y bajadas de 100–200 µV que duran alrededor de 1 s. Son movimientos de ojos.
- **F7 sigue al EOG** (cuando el EOG sube, F7 sube) y **F8 hace lo contrario** (cuando el EOG sube, F8 baja). Mirá por ejemplo alrededor de 29 s, 79 s y 86 s. **Ese es el ruido que queremos sacar.**
- **Fp1** también sigue al EOG.
- **Oz**, atrás de la cabeza y lejos de los ojos, no muestra nada de eso: es solo EEG. Por eso más adelante usamos Oz como "EEG limpio".
- **Fijate en los tamaños:** en el EOG los eventos llegan a ±200 µV, pero en F7 y F8 llegan solo a ±50 µV. A cada electrodo llega una *fracción* del ruido del ojo. Esa fracción (el `λ` de `y = s + λη`) es lo que el filtro tiene que aprender.

In [ ]:
# Promedio sincronizado con los eventos del EOG
VENTANA = int(0.5 * FS)


def promedio_sincronizado(x, picos, signo):
    """Promedio de x alrededor de cada evento (alineado en signo)."""
    tramos = [x[k - VENTANA:k + VENTANA] * s for k, s in zip(picos, signo) if VENTANA <= k < len(x) - VENTANA]
    return np.mean(tramos, axis=0)


picos, eog_lp = detectar_parpadeos(c["EOG"])
signo = np.sign(eog_lp[picos])
tau = np.arange(-VENTANA, VENTANA) / FS

fig, ax = plt.subplots(figsize=(10, 4))
for n in ["EOG", "F7", "F8", "Fp1", "Oz"]:
    ax.plot(tau, promedio_sincronizado(c[n], picos, signo), label=n)
ax.set_xlabel("t respecto del evento [s]")
ax.set_ylabel("µV")
ax.set_title(f"Promedio sincronizado con {len(picos)} eventos del EOG ({SES_EOG})")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Qué mirar: el promedio sincronizado

**Qué es:** en el gráfico anterior el ruido del ojo está mezclado con todo el EEG, y cuesta ver cuánto llega a cada canal. El truco es:
1. buscar los momentos en que el EOG tiene un pico grande (los "eventos");
2. recortar 1 s alrededor de cada uno;
3. promediar esos recortes. Antes se dan vuelta los que tenían el pico negativo, para que todos apunten para arriba.

Lo que está atado al evento (el ruido del ojo) se suma en todos los recortes. El EEG, que no tiene nada que ver con el evento, a veces está arriba y a veces abajo, y en el promedio tiende a 0.

**Qué mirar:**
- **EOG:** el evento mide unos 200 µV.
- **F7 y Fp1** suben unos 20–25 µV en el mismo momento, y **F8** baja unos 20 µV. O sea, a esos canales llega alrededor del 10 % del EOG, con signo opuesto a cada lado.
- **Oz** queda plano: no le llega nada.
- **Esta figura se repite en §6 con la salida del filtro.** Si el filtro anda, en `e` las curvas de F7 y F8 deberían quedar planas, como Oz.
- Son pocos eventos (6), así que el promedio es ruidoso.

In [ ]:
# ============================================================
# 2.2 PSD y EMG en los temporales
# ============================================================

fig, ax = plt.subplots(figsize=(10, 4))
for n in ["EOG", "Fp1", "Oz", "T9", "TP10"]:
    f, P = welch(DATOS["t01"][n], FS, nperseg=4 * FS)
    ax.semilogy(f, P, label=n, linewidth=0.8)
ax.set_xlabel("f [Hz]")
ax.set_ylabel("PSD [µV²/Hz]")
ax.set_title("PSD (t01)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Qué mirar: la PSD, y para qué la dibujamos

**Qué es la PSD (densidad espectral de potencia):** dice **cuánta potencia tiene la señal en cada frecuencia**. El eje vertical es logarítmico: cada línea de la grilla es 100 veces más que la de abajo.

**Por qué está:** para decidir cosas del preprocesado y para elegir qué canal usar como EEG limpio. Hay que mirar cuatro detalles:
1. **La caída en ~80 Hz.** Arriba de 80 Hz la potencia cae millones de veces: la grabación ya viene filtrada y ahí no hay nada. Por eso alcanza con 250 muestras por segundo (§1). Además, del EMG solo vemos la parte de 20–80 Hz; el EMG real llega a cientos de Hz.
2. **El valle en 50 Hz.** La red eléctrica (50 Hz) mete ruido. Acá aparece como un valle, no como un pico: ya le aplicaron un notch, así que no hace falta sacarlo.
3. **El bulto en ~10 Hz en Oz.** Es el **ritmo α**, actividad típica del cerebro en la parte de atrás de la cabeza. Confirma que Oz tiene EEG real, así que es un buen candidato para `s` en §4. Además, α es la banda que vigilamos en §6: el filtro no debería tocarla.
4. **El EOG por debajo de ~2 Hz.** El EOG tiene casi toda su potencia en frecuencias muy bajas, justo donde está la banda δ del EEG. Por eso un pasa-altos no serviría: se llevaría también el EEG lento.

In [ ]:
SES_EMG = tabla["kurt_emg_der"].idxmax()                      # Canción con más ráfagas de EMG
SES_CON_EMG = list(tabla.index[tabla["kurt_emg_der"] > 10])   # Canciones con ráfagas claras
print(f"Canción con más EMG: {SES_EMG}; canciones con ráfagas: {SES_CON_EMG}")

t = np.arange(len(EMG_DER[SES_EMG])) / FS + T_INI
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
axes[0].plot(t, EMG_IZQ[SES_EMG], linewidth=0.4)
axes[0].set_ylabel("EMG izq [µV]")
axes[1].plot(t, EMG_DER[SES_EMG], linewidth=0.4)
axes[1].set_ylabel("EMG der [µV]")
axes[2].plot(t, envolvente_hf(DATOS[SES_EMG]["Oz"]), label="Oz")
axes[2].plot(t, envolvente_hf(DATOS[SES_EMG]["T8"]), label="T8")
axes[2].set_ylabel("Envolvente >30 Hz")
axes[2].legend()
for ax in axes:
    ax.grid(True)
axes[0].set_title(f"Referencias de EMG (pasa-altos {F_HP_EMG} Hz), {SES_EMG}")
axes[-1].set_xlabel("t [s]")
plt.tight_layout()
plt.show()

### Qué mirar: las referencias de EMG (t15)

**Por qué está:** para comprobar que las referencias de EMG realmente "ven" músculo. Es la canción con más EMG.
- **Paneles 1 y 2, las referencias izquierda y derecha** (lo que está arriba de 25 Hz):
  - casi todo el tiempo son ruido parejo de ±10 µV;
  - en **~69 s y ~107 s** aparecen **ráfagas de ±50 µV**: son contracciones musculares;
  - aparecen a la vez de los dos lados.
- **Panel 3, la envolvente arriba de 30 Hz** en T8 (al lado de la sien derecha) y en Oz (la nuca). En las ráfagas, T8 crece mucho (de ~2 a ~25) y Oz poco (a ~7). El ruido es fuerte cerca de los músculos y débil lejos, como se espera del EMG.
- **Ojo:** fuera de las ráfagas, las referencias **no son cero**. Ese ruido parejo no es músculo: es EEG de alta frecuencia más el ruido del equipo, porque T9, T10, etc. también son electrodos de EEG. Si el filtro usa esa parte para restar, va a sacar cosas que no son EMG. Esto aparece en §5 y §6.

### Resumen de la inspección
- Los datos vienen **preprocesados**: no hay parpadeos típicos ni en el EOG ni en el EEG. Probablemente los sacaron (con ICA) antes de publicar el dataset.
- Queda ruido de **movimientos oculares horizontales** en F7, F8 y Fp1. La mejor canción para verlo es **t17**.
- Hay **EMG en ráfagas** en 4 de las 20 canciones, a la vez de los dos lados. La mejor es **t15**.
- **Oz** casi no tiene ninguno de los dos ruidos. Lo usamos como "EEG limpio" en §4 y como control en §6.
- **Consecuencia:** en los datos reales hay poco ruido para cancelar. Por eso los números confiables salen del semi-sintético (§4).

## §3. El filtro: NLMS y Wiener

> **Dónde estamos:** ya sabemos qué ruido hay. Ahora viene el bloque central del diagrama: el filtro `w` y la regla que lo ajusta.

### Qué entra al filtro
En cada instante `n`, el filtro mira las últimas M = 3 muestras de cada referencia y las pone en fila en un vector:

`x(n) = [EOG(n), EOG(n−1), EOG(n−2), EMGizq(n), EMGizq(n−1), EMGizq(n−2), EMGder(n), EMGder(n−1), EMGder(n−2)]`

Son 9 números. **¿Por qué varias muestras y no una sola?** Porque el ruido no llega al electrodo exactamente igual que a la referencia: puede llegar un poco atrasado o "suavizado". Con algunas muestras del pasado, el filtro puede armar esa versión.

### Qué hace el filtro
- `ŷ(n) = wᵀ x(n)`: una suma ponderada de esas 9 muestras. Es la estimación del ruido que hay en el electrodo.
- `e(n) = y(n) − ŷ(n)`: lo que queda. Es la estimación del EEG.

### Cómo aprende: NLMS (`7. Variantes de LMS.pdf`)

`w(n+1) = w(n) + μ̃ · x(n) · e(n) / (‖x(n)‖² + δ)`

**En palabras:** si `e(n)` quedó grande y "en la misma dirección" que `x(n)`, quedó ruido sin restar. Entonces se corrige `w` un poquito en esa dirección.
- **μ̃ (el paso):** cuánto corrige en cada muestra.
  - Grande: aprende rápido, pero "tiembla" (desajuste) y puede ponerse a seguir cosas que no debe.
  - Chico: más preciso, pero lento.
  - Es estable si 0 < μ̃ < 2.
- **La división por ‖x(n)‖²** es la "N" de NLMS. Hace que la corrección no dependa de qué tan fuerte esté la referencia en ese momento. Durante un movimiento de ojos `x` es enorme, y sin dividir (LMS común) daría un salto gigante. Por eso elegimos NLMS en la presentación.
- **δ:** un número chiquito para no dividir por cero cuando `x ≈ 0`.
- **w(0) = 0:** el filtro arranca sin restar nada.

### Wiener: lo mejor que puede hacer un filtro *fijo* (`3. Filtrado lineal óptimo.pdf`)
`wiener()` mira la grabación entera de una vez y calcula el `w` fijo que deja `e` lo más chico posible: `w_o = R⁻¹p`. No es adaptativo (necesita toda la señal de antemano), pero sirve de **regla para medir a NLMS**:
- **`J_min`:** el error que deja el mejor filtro fijo. NLMS no debería bajar mucho más que esto. Si baja, ojo: está siguiendo cosas que no debería.
- **bias:** qué tan lejos queda el `w` de NLMS del `w_o`.
- **Autovalores de R:** el aprendizaje de `w` tiene varios "modos", y cada autovalor dice qué tan rápido aprende uno. Si el **spread** (λmax/λmin) es grande, algunas partes de `w` aprenden muchísimo más lento que otras.

### Las otras funciones de la celda
- `matriz_regresores`: lo mismo que `armar_regresor`, pero para todos los `n` a la vez (la usa Wiener).
- `aportes`: separa `ŷ` en la parte que armó cada referencia (`ŷ_EOG`, `ŷ_EMG`), para poder graficarlas por separado.
- `rrmse(a, b)`: error relativo `√(promedio de (a − b)² / promedio de b²)`. 0 es que son iguales; 1 es que el error es tan grande como la señal.
- `rrmse_espectral`: lo mismo, pero comparando las PSD. Responde si las dos señales tienen la misma "forma" en frecuencia.

In [ ]:
def armar_regresor(refs, n, M):
    """x(n) = [x1(n), ..., x1(n-M+1), x2(n), ..., x2(n-M+1), ...]^T (ceros antes de n = 0)."""
    x = np.zeros(len(refs) * M)                      # Vector de entrada apilado, largo K*M
    for k, r in enumerate(refs):                     # Recorre cada referencia x_k
        for i in range(M):                           # Recorre los retardos 0, ..., M-1
            if n - i >= 0:                           # Antes de n = 0 la señal vale 0
                x[k * M + i] = r[n - i]              # x_k(n-i) en su lugar del vector apilado
    return x                                         # Devuelve x(n)


def nlms(y, refs, M, mu_t, delta):
    """
    NLMS con referencias apiladas.
    Devuelve:
        e: error (estimación del EEG limpio)
        y_hat: estimación del ruido en el electrodo
        W: W[n] = w(n), coeficientes usados en la muestra n
    """
    N = len(y)                                       # Cantidad de muestras
    w = np.zeros(len(refs) * M)                      # w(0) = 0
    W = np.zeros((N, len(refs) * M))                 # Historia de los coeficientes
    y_hat = np.zeros(N)                              # ŷ(n)
    e = np.zeros(N)                                  # e(n)
    for n in range(N):                               # Recorre las muestras
        x = armar_regresor(refs, n, M)               # Vector de entrada x(n)
        y_hat[n] = w @ x                             # ŷ(n) = w^H(n) x(n) (señales reales: w^H = w^T)
        e[n] = y[n] - y_hat[n]                       # e(n) = y(n) - ŷ(n)
        W[n] = w                                     # Guarda w(n)
        w = w + mu_t * x * e[n] / (x @ x + delta)    # w(n+1) = w(n) + μ̃ x(n) e*(n) / (||x(n)||² + δ)
    return e, y_hat, W


def matriz_regresores(refs, M):
    """Matriz N x (K*M) cuya fila n es x(n)^T."""
    N = len(refs[0])
    X = np.zeros((N, len(refs) * M))
    for k, r in enumerate(refs):
        for i in range(M):
            X[i:, k * M + i] = r[:N - i]
    return X


def wiener(y, refs, M):
    """Solución de Wiener estimada con promedios temporales."""
    X = matriz_regresores(refs, M)
    R = X.T @ X / len(y)                             # R = E{x x^T}
    p = X.T @ y / len(y)                             # p = E{x y}
    w_o = np.linalg.solve(R, p)                      # w_o = R^-1 p
    lam = np.linalg.eigvalsh(R)
    return {"w_o": w_o, "J_min": np.mean((y - X @ w_o)**2), "autovalores": lam,
            "spread": lam.max() / lam.min(), "P_x": np.trace(R)}


def aportes(refs, W, M):
    """Parte de ŷ que arma cada referencia: ŷ_k(n) = w_k(n)^T x_k(n)."""
    X = matriz_regresores(refs, M)
    return [np.sum(W[:, k * M:(k + 1) * M] * X[:, k * M:(k + 1) * M], axis=1) for k in range(len(refs))]


def rrmse(a, b):
    return np.sqrt(np.mean((a - b)**2) / np.mean(b**2))


def rrmse_espectral(a, b):
    _, Pa = welch(a, FS, nperseg=FS)
    _, Pb = welch(b, FS, nperseg=FS)
    return rrmse(Pa, Pb)

## §4. Experimento semi-sintético (Camino A: con ground truth)

> **Dónde estamos:** el filtro está programado. Antes de creerle en datos reales, queremos saber **si funciona y con qué parámetros**. Para eso necesitamos un caso en el que conozcamos `s`.

### Cómo se arma la mezcla
Es el mismo diagrama de antes, pero con `s` elegido por nosotros:
1. **`s(n)`** = 20 s de **Oz** de una canción al azar. Elegimos Oz porque en §2 vimos que casi no tiene ruido de ojos ni de músculo.
2. **`η_EOG`** = 20 s del canal EOG de *otra* canción. Tiene que ser otra para que no tenga nada que ver con `s`: es lo que supone la teoría.
3. **`η_EMG`** = 20 s de la referencia de EMG derecha de una canción con ráfagas.
4. **`y = s + λ₁η_EOG + λ₂η_EMG`**, con los `λ` elegidos para lograr una relación señal a ruido (**SNR**) dada.
   - SNR en dB = 10·log₁₀(potencia de `s` / potencia del ruido).
   - SNR = 0 dB: EEG y ruido son igual de fuertes.
   - SNR = −5 dB: el ruido tiene unas 3 veces más potencia que el EEG.
5. **Referencias `x₁`, `x₂`:** no le damos al filtro el `η` exacto, porque sería demasiado fácil. En la realidad, la referencia nunca mide exactamente lo mismo que llega al electrodo. Lo simulamos con `x = h∗η + ruido`:
   - `h` es un filtrito corto: `[1, 0.5]` para el EOG y `[1, −0.4]` para el EMG;
   - el ruido de medición está 20 dB por debajo de la referencia.
6. **Monte Carlo:** repetimos todo 30 veces, con canciones y tramos distintos, y promediamos. Un caso solo puede salir bien o mal de casualidad.

### Qué medimos (lo prometido en la design review)
- **Curva de aprendizaje `J(n)`:** el promedio de `e(n)²` sobre las 30 repeticiones, en cada instante. Muestra cómo baja el error mientras el filtro aprende.
- **`J_min`:** el error del mejor filtro fijo (Wiener). **No es 0**, porque `e` sigue conteniendo a `s`. Para el filtro el EEG es "error", pero es justo lo que queremos que quede. Lo ideal es `J(∞) ≈ J_min ≈ potencia de s`.
- **Desajuste `ℳ = (J(∞) − J_min) / J_min`:** cuánto error de más deja NLMS respecto del mejor filtro fijo, por estar siempre corrigiendo.
- **`t_conv`:** cuánto tarda en llegar a régimen.
- **Sesgo (bias):** `‖promedio de w al final − w_o‖`.
- **`RRMSE_y` y `RRMSE_e`:** cuánto se parecen a `s` la entrada `y` (cuánto ruido había antes) y la salida `e` (cuánto queda después). Si `RRMSE_e < RRMSE_y`, el filtro limpió.
- **`CC`:** correlación entre `e` y `s` (1 = idénticas).
- **`RRMSE_esp`:** lo mismo que RRMSE, pero comparando las PSD.

In [ ]:
L_TRAMO = 20 * FS
H1 = np.array([1.0, 0.5])       # Camino η_EOG -> x1
H2 = np.array([1.0, -0.4])      # Camino η_EMG -> x2
SNR_REF_DB = 20                 # SNR de las referencias (ruido de medición)
DELTA_REL = 1e-3                # δ relativo a la potencia media de ||x||²


def escalar_por_snr(s, eta, snr_db):
    return np.sqrt(np.mean(s**2) / (np.mean(eta**2) * 10**(snr_db / 10)))


def tramo_al_azar(x, rng, L=L_TRAMO):
    i0 = rng.integers(0, len(x) - L)
    return x[i0:i0 + L]


def con_ruido(x, snr_db, rng):
    return x + rng.normal(0, np.sqrt(np.mean(x**2) / 10**(snr_db / 10)), len(x))


def realizacion(rng, snr_eog, snr_emg, h1=H1, h2=H2, snr_ref_db=SNR_REF_DB, canal_s="Oz"):
    ses_s, ses_eog = rng.choice(SESIONES, 2, replace=False)
    ses_emg = rng.choice([x for x in SES_CON_EMG if x != ses_s])
    s = tramo_al_azar(DATOS[ses_s][canal_s], rng)
    eta1 = tramo_al_azar(DATOS[ses_eog]["EOG"], rng)
    eta2 = tramo_al_azar(EMG_DER[ses_emg], rng)
    y = s + escalar_por_snr(s, eta1, snr_eog) * eta1 + escalar_por_snr(s, eta2, snr_emg) * eta2
    x1 = con_ruido(lfilter(h1, 1, eta1), snr_ref_db, rng)
    x2 = con_ruido(lfilter(h2, 1, eta2), snr_ref_db, rng)
    return s, y, [x1, x2]


def suavizar(x, largo=FS // 2):
    """Promedio móvil de medio segundo (en los bordes promedia solo las muestras que hay)."""
    ventana = np.ones(largo)
    return np.convolve(x, ventana, mode="same") / np.convolve(np.ones(len(x)), ventana, mode="same")


def montecarlo(mu_t, M, snr_eog, snr_emg, N_real=30, semilla=1, **kw):
    rng = np.random.default_rng(semilla)
    J = np.zeros(L_TRAMO)                            # Curva de aprendizaje J(n) = E{|e(n)|²}
    dif_w, J_min, P_s, met = [], [], [], []
    regimen = slice(L_TRAMO // 2, L_TRAMO)           # Segunda mitad: régimen permanente
    for _ in range(N_real):
        s, y, refs = realizacion(rng, snr_eog, snr_emg, **kw)
        wi = wiener(y, refs, M)
        delta = DELTA_REL * wi["P_x"]
        e, _, W = nlms(y, refs, M, mu_t, delta)
        J += e**2 / N_real
        dif_w.append(W[-1] - wi["w_o"])
        J_min.append(wi["J_min"])
        P_s.append(np.mean(s**2))
        met.append([rrmse(y[regimen], s[regimen]), rrmse(e[regimen], s[regimen]),
                    rrmse_espectral(e[regimen], s[regimen]), np.corrcoef(e[regimen], s[regimen])[0, 1]])
    J_inf = J[regimen].mean()
    J_min = np.mean(J_min)
    Js = suavizar(J)
    n_conv = np.argmax(Js <= 1.1 * J_inf)            # Primera muestra a menos de 10 % de J(∞)
    met = np.mean(met, axis=0)
    return {"J": J, "J_min": J_min, "P_s": np.mean(P_s), "J_inf": J_inf,
            "desajuste": (J_inf - J_min) / J_min, "t_conv": n_conv / FS,
            "sesgo": np.linalg.norm(np.mean(dif_w, axis=0)),
            "RRMSE_y": met[0], "RRMSE_e": met[1], "RRMSE_esp": met[2], "CC": met[3]}

In [ ]:
# ============================================================
# 4.0 Chequeos de cordura
# ============================================================

M = 3
rng = np.random.default_rng(0)

# (a) Caso de libro: referencias blancas, y = s + w_true^T x  ->  w(N) tiene que llegar a w_o
N = 20000
x1, x2 = rng.normal(size=N), rng.normal(size=N)
s = 0.1 * rng.normal(size=N)
y = s + lfilter([0.8, -0.3, 0.1], 1, x1) + lfilter([0.5, 0.2, 0.0], 1, x2)
_, _, W = nlms(y, [x1, x2], M, 0.1, 1e-6)
print("(a) w(N) =", W[-1].round(3), "  w_o =", wiener(y, [x1, x2], M)["w_o"].round(3))

# (b) Referencia exacta pero con el EOG real: el spread de R hace que el modo lento tarde mucho
r = montecarlo(0.1, M, -5, 0, N_real=5, h1=np.array([1.0]), h2=np.array([1.0]), snr_ref_db=np.inf)
s, y, refs = realizacion(np.random.default_rng(3), -5, 0)
print(f"(b) Referencia exacta: RRMSE(y,s) = {r['RRMSE_y']:.2f} -> RRMSE(e,s) = {r['RRMSE_e']:.2f} "
      f"(J_min/P_s = {r['J_min'] / r['P_s']:.3f}: Wiener sí limpia)")
print(f"    spread solo EOG = {wiener(y, refs[:1], M)['spread']:.0f}, "
      f"spread apilado [EOG, EMG] = {wiener(y, refs, M)['spread']:.0f}")

# (c) μ̃ > 2 tiene que divergir
with np.errstate(all="ignore"):
    r = montecarlo(2.5, M, -5, 0, N_real=3)
print(f"(c) μ̃ = 2.5:  J(∞) = {r['J_inf']:.3g}  (nan/inf: diverge)")

### Qué mirar: los chequeos de cordura

**Por qué están:** antes de interpretar resultados hay que comprobar que el código hace lo que tiene que hacer.

- **(a) Caso de libro.** Referencias de ruido blanco y un camino conocido. NLMS tiene que encontrar los coeficientes verdaderos, y da `w(N) ≈ w_o`. **El código de NLMS está bien.**
- **(b) Referencia perfecta, pero con el EOG real.** Le damos al filtro el ruido exacto. Wiener limpia casi perfecto (`J_min / P_s ≈ 1`: solo queda el EEG), pero NLMS no llega en 20 s (RRMSE 0.74). ¿Por qué, si el código está bien? Por el **spread de autovalores**.
  - **Intuición:** el EOG es muy lento, así que `EOG(n)`, `EOG(n−1)` y `EOG(n−2)` son casi iguales. Para el filtro es casi imposible distinguir "poner peso en `EOG(n)`" de "poner peso en `EOG(n−1)`", y esa dirección de `w` aprende lentísimo. El spread mide justamente eso: da ~300 con el EOG solo.
  - **Al apilar EOG y EMG**, el spread sube a ~14 000. Las referencias tienen potencias muy distintas (el EOG es mucho más fuerte), y NLMS divide por la potencia *total*, que manda el EOG. Entonces los coeficientes del EMG reciben pasos chiquitos y aprenden lento.
  - Esto es lo que dijimos en la presentación: **NLMS normaliza la potencia, pero no arregla el spread.**
- **(c) Con μ̃ = 2.5 el filtro diverge** (da nan), como dice la teoría: tiene que ser 0 < μ̃ < 2.

In [ ]:
# ============================================================
# 4.1 Barrido de μ̃: curva de aprendizaje, desajuste, velocidad, bias
# ============================================================

M = 3
SNR_EOG, SNR_EMG = -5, 0
MUS = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]

res_mu = {mu: montecarlo(mu, M, SNR_EOG, SNR_EMG) for mu in MUS}

n = np.arange(L_TRAMO) / FS
fig, ax = plt.subplots(figsize=(12, 5))
for mu, r in res_mu.items():
    ax.plot(n, 10 * np.log10(suavizar(r["J"]) / r["P_s"]), label=f"μ̃ = {mu}")
ax.axhline(10 * np.log10(res_mu[MUS[0]]["J_min"] / res_mu[MUS[0]]["P_s"]), color="k", linestyle="--", label="J_min")
ax.set_xlabel("t [s]")
ax.set_ylabel("J(n) / P_s [dB]")
ax.set_title(f"Curva de aprendizaje NLMS (M = {M}, SNR_EOG = {SNR_EOG} dB, SNR_EMG = {SNR_EMG} dB, 30 realizaciones)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

tabla_mu = pd.DataFrame({mu: {k: v for k, v in r.items() if k != "J"} for mu, r in res_mu.items()}).T
tabla_mu.index.name = "μ̃"
print(tabla_mu.round(3).to_string())

### Qué mirar: la curva de aprendizaje y la tabla

**Por qué están:** es el gráfico central del TP. Muestra cómo aprende el filtro y cuánto error deja, y sirve para **elegir μ̃**.
- **Eje vertical:** `J(n)` dividido por la potencia del EEG, en dB. **0 dB quiere decir que el error es del tamaño del EEG**, y eso es lo mejor que tiene sentido, porque `e` debería *ser* el EEG.
- **Línea punteada:** `J_min` (Wiener), ~0.1 dB.
- **En t = 0** el filtro no sabe nada (`w = 0`), así que `e = y`, todo ruido. Con el tiempo el error baja.
- **μ̃ chico (0.003 y 0.01):** baja lento, y en 20 s no llega a `J_min`. Le falta tiempo.
- **μ̃ = 0.03:** llega a `J_min` hacia los 11–12 s, y desde ahí se queda dando vueltas alrededor de la línea.
- **μ̃ grande (0.1 a 1):** baja rápido y queda **por debajo** de `J_min`. Parece mejor, pero no lo es.
  - `J_min` es lo que deja el mejor filtro *fijo*. Uno que corrige muy rápido persigue cada detalle y empieza a **restar EEG**.
  - Mirá la tabla: con μ̃ = 1, `J_inf` es el más chico, pero `RRMSE_e` = 1.10 y `CC` = 0.24. O sea, `e` se parece muy poco a `s`.
  - **Que el error sea chico no quiere decir que el EEG esté limpio.** Por eso medimos contra `s` y no solo el MSE. El "desajuste" negativo de la tabla viene de lo mismo.
- **Mejor compromiso: μ̃ ≈ 0.03.**
  - `RRMSE_e` = 0.67, contra 2.0 sin filtrar.
  - `CC` = 0.78.
  - Desajuste ≈ 8 %.

In [ ]:
# ============================================================
# 4.2 Largo del filtro M (solo Wiener) y modos
# ============================================================

rng = np.random.default_rng(2)
ORDENES = range(1, 17)
J_M, spread_M = np.zeros(len(ORDENES)), np.zeros(len(ORDENES))
for _ in range(10):
    s, y, refs = realizacion(rng, SNR_EOG, SNR_EMG)
    for j, m in enumerate(ORDENES):
        wi = wiener(y, refs, m)
        J_M[j] += wi["J_min"] / np.mean(s**2) / 10
        spread_M[j] += wi["spread"] / 10

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(ORDENES, 10 * np.log10(J_M), "o-")
axes[0].set_xlabel("M")
axes[0].set_ylabel("J_min / P_s [dB]")
axes[0].set_title("Piso de Wiener vs largo del filtro")
axes[1].semilogy(ORDENES, spread_M, "o-")
axes[1].set_xlabel("M")
axes[1].set_ylabel("λmax / λmin")
axes[1].set_title("Spread de autovalores de R")
for ax in axes:
    ax.grid(True)
plt.tight_layout()
plt.show()

### Qué mirar: ¿cuántos coeficientes M?

**Por qué está:** M es otro parámetro que hay que justificar. Con más coeficientes el filtro puede armar mejor el ruido, pero también aprende más lento.
- **Izquierda, `J_min` (lo mejor posible) contra M:** baja de M = 1 a M = 3, y después casi no baja más. Más de 3 coeficientes no ayudan.
- **Derecha, spread contra M** (escala logarítmica): crece de ~10³ a ~10⁶. Cada coeficiente de más agrega un modo todavía más lento.
- **Conclusión: M = 3**, igual que He et al. (2004).

In [ ]:
# ============================================================
# 4.3 Barrido de SNR
# ============================================================

MU_ELEGIDO = 0.03     # Ajustar mirando la tabla de 4.1 (compromiso velocidad / desajuste)
SNRS = [-10, -5, 0, 5, 10]
res_snr = [montecarlo(MU_ELEGIDO, M, snr, snr, N_real=20) for snr in SNRS]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(SNRS, [r["RRMSE_y"] for r in res_snr], "o-", label="sin filtrar: RRMSE(y, s)")
ax.plot(SNRS, [r["RRMSE_e"] for r in res_snr], "o-", label="NLMS: RRMSE(e, s)")
ax.plot(SNRS, [r["RRMSE_esp"] for r in res_snr], "o-", label="NLMS: RRMSE espectral")
ax.set_xlabel("SNR de entrada (EOG y EMG) [dB]")
ax.set_ylabel("RRMSE")
ax.set_title(f"Limpieza vs SNR (μ̃ = {MU_ELEGIDO}, M = {M})")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Qué mirar: cuánto ayuda el filtro según cuánto ruido hay

**Por qué está:** para ver en qué condiciones conviene usar el filtro. A la izquierda del eje hay mucho ruido; a la derecha, poco.
- **Azul, RRMSE de `y`** (sin filtrar): cuánto ruido había.
- **Naranja, RRMSE de `e`** (con el filtro): cuánto queda.
- **Con mucho ruido (−10 dB)** el filtro mejora muchísimo: de 4.4 a 1.4.
- **Con poco ruido (+10 dB)** el filtro queda apenas **peor** que no filtrar (0.53 contra 0.45). Cuando casi no hay ruido para sacar, pesa más el error propio del filtro (el desajuste y el EEG que resta) que lo que limpia.
- **Verde, RRMSE espectral:** es bajo (0.2–0.4 desde −5 dB). La forma de la PSD de `e` se parece mucho a la de `s`, aunque la forma de onda en el tiempo no sea idéntica.

## §5. Datos reales (Camino B)

> **Dónde estamos:** el filtro funciona en el caso armado (§4), y elegimos μ̃ = 0.03 y M = 3. Ahora lo usamos tal cual el diagrama, sobre grabaciones crudas.

- `y` = un canal de EEG tal como se grabó.
- `x = [EOG, EMG izq, EMG der]`.
- `e` = la salida del filtro: lo que esperamos que sea el EEG limpio.
- `ŷ_EOG`, `ŷ_EMG` = lo que el filtro decidió que era ruido de ojos y de músculo, o sea lo que restó. Siempre se cumple `y = e + ŷ_EOG + ŷ_EMG`.

**Acá no conocemos `s`**, así que no podemos calcular el RRMSE. En esta sección solo miramos qué pasa; en §6 lo evaluamos.

Usamos dos canciones:
- **t17**, la de más ruido de ojos (§2): F7, F8 y Fp1 (cerca de los ojos), y Oz (lejos).
- **t15**, la de más EMG: T8 (sien derecha), F8, Fp1 y Oz.

In [ ]:
M = 3
NOMBRES_REF = ["EOG", "EMG izq", "EMG der"]


def procesar_real(ses, canal, mu_t=MU_ELEGIDO, M=M):
    c = DATOS[ses]
    refs = [c["EOG"], EMG_IZQ[ses], EMG_DER[ses]]
    delta = DELTA_REL * sum(np.mean(r**2) for r in refs) * M
    e, y_hat, W = nlms(c[canal], refs, M, mu_t, delta)
    return {"y": c[canal], "e": e, "W": W, "refs": refs, "aportes": aportes(refs, W, M),
            "wiener": wiener(c[canal], refs, M)}


CASOS = [(SES_EOG, n) for n in ["F7", "F8", "Fp1", "Oz"]] + [(SES_EMG, n) for n in ["T8", "F8", "Fp1", "Oz"]]
reales = {caso: procesar_real(*caso) for caso in CASOS}


def graficar_real(ses, canal):
    r = reales[(ses, canal)]
    t = np.arange(len(r["y"])) / FS + T_INI
    fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
    for ax, x, nombre in zip(axes, [r["y"], r["e"], r["aportes"][0], r["aportes"][1] + r["aportes"][2]],
                             ["y", "e", "ŷ_EOG", "ŷ_EMG"]):
        ax.plot(t, x, linewidth=0.4)
        ax.set_ylabel(f"{nombre} [µV]")
        ax.grid(True)
    axes[0].set_title(f"{canal}, {ses}: entrada, salida y ruido estimado")
    axes[-1].set_xlabel("t [s]")
    plt.tight_layout()
    plt.show()

In [ ]:
graficar_real(SES_EOG, "F8")

### Qué mirar: F8 en t17 (ruido de ojos)

**Por qué está:** es el caso con más ruido de ojos. De arriba a abajo: lo que entra (`y`), lo que sale (`e`), y lo que el filtro restó (`ŷ_EOG` y `ŷ_EMG`).
- **`ŷ_EOG` tiene la forma de los movimientos de ojos:** ondas lentas de alrededor de 1 s. Donde `y` tiene una onda grande (por ejemplo en 79 s, 87 s y 96 s), `e` queda más chato. **Ahí sí sacó ruido de ojos.**
- **Problema 1:** en ~70.5 s, `ŷ_EOG` tiene un pico de +125 µV que en `y` no estaba, y por eso `e` muestra un pico de −110 µV. El filtro **restó de más y creó un artefacto que antes no existía**. Coincide con un salto muy brusco del EOG (−240 µV, se ve en §2.1). Ese movimiento es muy distinto de los demás, y los `w` que el filtro aprendió con el resto no le sirven.
- **Problema 2:** `ŷ_EMG` es ±10 µV de ruido parejo *todo el tiempo*, aunque en t17 no hay ráfagas de músculo. El filtro usa la parte pareja de las referencias de EMG, que no es músculo, para restarle alta frecuencia a F8. **Lo más probable es que estemos sacando EEG.**

In [ ]:
graficar_real(SES_EMG, "T8")

### Qué mirar: T8 en t15 (ruido de músculo)

**Por qué está:** es el caso con más EMG, en el electrodo más cercano a la sien derecha.
- `y` tiene ráfagas de músculo en ~69 s y ~107 s.
- **`ŷ_EMG` tiene esas mismas ráfagas:** el filtro reconoce el EMG y lo resta. En `e`, la ráfaga de 69 s prácticamente desaparece; la de 107 s se achica, pero no del todo.
- `ŷ_EMG` arranca en cero y crece durante los primeros ~10 s: es el filtro aprendiendo, porque `w` arranca en 0.
- **Problema:** otra vez, `ŷ_EOG` mete picos que en `y` no estaban (en 23 s, +60 µV, y en 77 s), y `e` queda con picos negativos ahí. Es lo mismo que en F8: algunos movimientos de ojos no pasan del EOG al canal igual que los demás.

## §6. Validación sin ground truth: ¿le creemos a `e(n)`?

> **Dónde estamos:** el filtro ya corrió sobre los datos reales (§5). Ahora queremos decidir si lo que salió es EEG limpio.

**Recordatorio de qué es cada cosa:**
- **`s(n)`:** el EEG limpio. Es lo que queremos, pero **en datos reales no lo conocemos**.
- **`y(n)`:** lo que se grabó, o sea `s` + ruido de ojos + ruido de músculo.
- **`e(n)`:** la salida del filtro, `y` menos lo que el filtro restó. **Queremos que `e = s`.**

Como no tenemos `s`, no podemos medir directamente qué tan cerca está `e` de `s`. Lo que sí podemos hacer es preguntarnos **"si `e` fuera de verdad `s`, ¿qué tendría que cumplirse?"** y comprobarlo. Cada prueba es una de esas consecuencias. Si `e = s`, entonces:

1. **El ruido de ojos desapareció de `e`,** porque `s` no tiene ruido de ojos.
   - *Cómo se mide:* se repite el promedio sincronizado de §2.1 sobre `e`. El evento en F7 y F8 tendría que quedar plano, como en Oz.
   - También tendría que bajar la potencia en 0.5–4 Hz (donde está el EOG) en F7 y F8, y no en Oz.
2. **El EMG desapareció de `e` durante las ráfagas,** porque `s` no tiene músculo.
   - *Cómo se mide:* la potencia entre 30 y 80 Hz durante las ráfagas tiene que bajar.
3. **Fuera de las ráfagas, la alta frecuencia no cambió.** Ahí no hay músculo, así que lo que hay entre 30 y 80 Hz es EEG (`s`), y el filtro no debería tocarlo. *Si baja, el filtro está sacando EEG.*
4. **El ritmo α (8–13 Hz) no cambió.** α es EEG: ni los ojos ni los músculos tienen mucho ahí. *Si cambia, el filtro está tocando `s`.*
5. **Oz casi no cambió** (control negativo). Oz casi no tiene ruido, así que ahí `y ≈ s`. Si el filtro anda bien, tiene que dar `e ≈ y`. *Si saca mucho de Oz, está sacando EEG.*
6. **Lejos de la fuente de ruido se saca poco.** El ruido de ojos llega fuerte adelante y débil atrás; el de músculo, fuerte cerca de la sien. Esto se mira en §6.5.
7. **El `w` de NLMS quedó cerca del de Wiener.**
8. **`e` no se parece a las referencias** (correlación ≈ 0). Esta es la prueba más débil: el filtro siempre la cumple, porque es justo lo que minimiza. Solo dice que el filtro convergió, no que limpió bien.

**Cómo leer los dB de la tabla:** cada número es cuánto cambió la potencia de `e` respecto de la de `y`.
- 0 dB: no cambió.
- −3 dB: quedó la mitad.
- −10 dB: quedó la décima parte.
- Negativo, en general: se sacó algo.

In [ ]:
def potencia_banda(x, mascara, f1, f2):
    f, P = welch(x[mascara], FS, nperseg=FS)
    return P[(f >= f1) & (f < f2)].sum()


def db(a, b):
    return 10 * np.log10(a / b)


filas = []
for (ses, canal), r in reales.items():
    env = envolvente_hf(EMG_DER[ses])
    rafaga, calma = env > np.percentile(env, 90), env < np.percentile(env, 50)
    todo = np.ones(len(r["y"]), dtype=bool)
    fila = {"caso": f"{ses} {canal}",
            "total": db(np.mean(r["e"]**2), np.mean(r["y"]**2)),
            "0.5-4 Hz": db(potencia_banda(r["e"], todo, 0.5, 4), potencia_banda(r["y"], todo, 0.5, 4)),
            "α": db(potencia_banda(r["e"], todo, 8, 13), potencia_banda(r["y"], todo, 8, 13)),
            "30-80 ráfaga": db(potencia_banda(r["e"], rafaga, 30, 80), potencia_banda(r["y"], rafaga, 30, 80)),
            "30-80 calma": db(potencia_banda(r["e"], calma, 30, 80), potencia_banda(r["y"], calma, 30, 80)),
            "‖w(N)−w_o‖/‖w_o‖": np.linalg.norm(r["W"][-1] - r["wiener"]["w_o"]) / np.linalg.norm(r["wiener"]["w_o"])}
    for k, nombre in enumerate(NOMBRES_REF):
        fila[f"corr(y,{nombre})"] = np.corrcoef(r["y"], r["refs"][k])[0, 1]
        fila[f"corr(e,{nombre})"] = np.corrcoef(r["e"], r["refs"][k])[0, 1]
    filas.append(fila)
print("Cambio de potencia de e respecto de y [dB] (negativo = se sacó potencia):")
print(pd.DataFrame(filas).set_index("caso").round(2).T.to_string())

### Qué mirar en la tabla

Cada columna es una combinación de canción y canal: t17 es el caso de ruido de ojos y t15 el de músculo.
- **Prueba 4, fila `α`:** entre −0.05 y +0.33 dB en todos los casos. **α casi no cambia ✔**: el filtro no toca el ritmo α.
- **Prueba 2, fila `30-80 ráfaga`:** en T8 (t15) da −18.6 dB, o sea que quedó 1/70 de la potencia. **El EMG de las ráfagas se va ✔.**
- **Prueba 3, fila `30-80 calma`:** debería dar ≈ 0, pero da unos −8 dB en T8 y F8, y −7.5 dB en F7. **✘ Se saca alta frecuencia donde no hay músculo.** Muy probablemente es EEG (β/γ), que también está en las referencias de EMG porque T9, T10, etc. son electrodos de EEG. Es lo mismo que se vio en `ŷ_EMG` de F8 (§5).
- **Prueba 5, fila `total` en Oz:** −0.4 dB en t17 **✔** (casi nada). En t15 da −1.0 dB: un poco más, porque las ráfagas también llegan algo a Oz.
- **Prueba 1, fila `0.5-4 Hz`:** en t17 baja −1.5 dB en F7 y F8, y solo −0.6 dB en Oz, lo que va en la dirección correcta. Pero en t15 Oz también baja −1.5 dB. **No es concluyente.**
- **Prueba 7, fila `‖w(N)−w_o‖/‖w_o‖`:** da 0.4–0.65. El `w` final de NLMS queda **a ~50 % del de Wiener ✘**. Hay dos causas: los modos lentos (§4, chequeo b) y que la relación entre el ruido y el electrodo cambia con el tiempo.
- **Prueba 8, filas `corr(y,·)` contra `corr(e,·)`:** antes, `y` se parecía hasta 0.6 al EOG; después, `e` se parece menos de 0.1 a todas las referencias **✔**. Pero, como se dijo, esto solo prueba que el filtro minimizó, no que `e` sea `s`.

In [ ]:
# Promedio sincronizado con los eventos del EOG: y vs e
picos, eog_lp = detectar_parpadeos(DATOS[SES_EOG]["EOG"])
signo = np.sign(eog_lp[picos])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, canal in zip(axes, ["F7", "F8", "Oz"]):
    r = reales[(SES_EOG, canal)]
    ax.plot(tau, promedio_sincronizado(r["y"], picos, signo), label="y")
    ax.plot(tau, promedio_sincronizado(r["e"], picos, signo), label="e")
    ax.set_title(f"{canal} ({SES_EOG}), {len(picos)} eventos")
    ax.set_xlabel("t respecto del evento [s]")
    ax.legend()
    ax.grid(True)
axes[0].set_ylabel("µV")
plt.tight_layout()
plt.show()

### Qué mirar: ¿desapareció el ruido de ojos? (prueba 1)

**Por qué está:** es la misma figura que en §2.1, pero comparando antes del filtro (`y`, azul) y después (`e`, naranja). Si `e = s`, la curva naranja de F7 y F8 tendría que quedar plana, como en Oz.
- **No queda plana: queda dada vuelta ✘.** En F7, antes del filtro el evento subía unos +20 µV; después baja unos −15 µV. En F8, antes −20 µV y después +18 µV. **El filtro restó más ruido de ojos del que había.**
- **Oz:** antes y después iguales ✔. No había nada y no se tocó.
- **¿Por qué resta de más?** Hay un único canal EOG y un único `w` para todos los movimientos. Pero no todos los movimientos de ojos llegan igual a F7 y F8: mirar a un costado, mirar arriba o parpadear tienen cada uno su propio "camino". El `w` que sirve en promedio para toda la grabación no sirve para estos 6 eventos grandes.
- **Wiener tampoco lo saca bien**, y es el mejor filtro fijo. Así que el problema no es NLMS: **una sola referencia de EOG no alcanza**. En la literatura se usan dos, una vertical y una horizontal.
- Son solo 6 eventos, así que el promedio es ruidoso.

In [ ]:
# PSD y vs e durante ráfagas de EMG
env = envolvente_hf(EMG_DER[SES_EMG])
rafaga = env > np.percentile(env, 90)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, canal in zip(axes, ["T8", "Oz"]):
    for nombre in ["y", "e"]:
        f, P = welch(reales[(SES_EMG, canal)][nombre][rafaga], FS, nperseg=FS)
        ax.semilogy(f, P, label=nombre)
    ax.set_title(f"{canal} ({SES_EMG}): PSD durante ráfagas de EMG")
    ax.set_xlabel("f [Hz]")
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()

### Qué mirar: ¿desapareció el EMG? (prueba 2)

**Por qué está:** compara la PSD antes (`y`) y después (`e`) del filtro, **solo en los momentos de ráfagas** de t15. Si `e = s`, la alta frecuencia que agregó el músculo debería desaparecer y lo demás quedar igual.
- **T8 (sien derecha):**
  - entre 20 y 80 Hz, `e` queda unas 10 veces por debajo de `y` **✔**: el filtro sacó la potencia del músculo;
  - por debajo de 15 Hz, `y` y `e` se superponen **✔**: no tocó el EEG lento.
- **Ojo con el pico de 50 Hz:** durante las ráfagas, `y` tiene un pico grande en 50 Hz (la frecuencia de la red eléctrica), en T8 y también en Oz.
  - El EMG es de banda ancha, no un pico en 50 Hz.
  - Lo más probable es que, cuando la persona aprieta la mandíbula o se mueve, los electrodos se muevan y capten la red eléctrica.
  - O sea, **parte de lo que llamamos "ráfagas de EMG" es interferencia de línea por movimiento**. El filtro la saca igual, porque también está en las referencias, pero hay que decirlo en el informe.
- **Oz:** salvo el pico de 50 Hz, `y` y `e` son iguales **✔**.

In [ ]:
# ============================================================
# 6.5 Plausibilidad física: potencia removida vs distancia a la fuente (Wiener, todos los canales)
# ============================================================

pos = electrodos.set_index("name")[["x", "y", "z"]]
RPA = np.array([0.0825, 0.0, 0.0])   # Punto preauricular derecho (coordsystem.json)
NAS = np.array([0.0, 0.114, 0.0])    # Nasión (cerca de los ojos)

filas = []
for canal in CANALES_EEG:
    if canal in CANALES_REF:
        continue
    p = pos.loc[canal].to_numpy()
    y_eog, y_emg = DATOS[SES_EOG][canal], DATOS[SES_EMG][canal]
    filas.append({"canal": canal, "d_NAS": np.linalg.norm(p - NAS), "d_RPA": np.linalg.norm(p - RPA),
                  "aten_EOG_dB": db(np.mean(y_eog**2), wiener(y_eog, [DATOS[SES_EOG]["EOG"]], M)["J_min"]),
                  "aten_EMG_dB": db(np.mean(y_emg**2), wiener(y_emg, [EMG_DER[SES_EMG]], M)["J_min"])})
espacial = pd.DataFrame(filas).set_index("canal")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, d, aten, titulo in [(axes[0], "d_NAS", "aten_EOG_dB", f"Referencia EOG ({SES_EOG})"),
                            (axes[1], "d_RPA", "aten_EMG_dB", f"Referencia EMG der ({SES_EMG})")]:
    ax.scatter(espacial[d] * 100, espacial[aten])
    for canal in espacial[aten].nlargest(5).index:
        ax.annotate(canal, (espacial.loc[canal, d] * 100, espacial.loc[canal, aten]))
    ax.set_title(titulo)
    ax.set_ylabel("Potencia removida [dB]")
    ax.grid(True)
axes[0].set_xlabel("Distancia al nasión [cm]")
axes[1].set_xlabel("Distancia a RPA [cm]")
plt.tight_layout()
plt.show()

### Qué mirar: ¿se saca más cerca de la fuente? (prueba 6)

**Por qué está:** si lo que resta el filtro es de verdad ruido de ojos (o de músculo), tiene que ser más fuerte cerca de los ojos (o de la sien) y casi cero lejos. Si sacara lo mismo en todos lados, no estaría sacando el ruido de una fuente concreta. Acá se usa Wiener, que es más rápido, sobre los 60 canales de EEG.
- **Eje horizontal:** la distancia de cada electrodo a la fuente: al nasión (entre los ojos) para el EOG, y al punto delante de la oreja derecha para el EMG.
- **Eje vertical:** cuánta potencia se saca, en dB.
- **EOG (izquierda):** los canales a 5–12 cm de los ojos pierden ~2 dB, y los que están a 20 cm (la nuca) ~0 dB. **Baja con la distancia ✔**: lo que se saca parece ruido de ojos de verdad.
- **EMG derecho (derecha):** T8, el más cercano, pierde ~0.6 dB, pero AF3 y AF7, a 17 cm y del otro lado de la cabeza, pierden lo mismo. **No baja con la distancia ✘.**
  - Entonces lo que saca la referencia de EMG no es solo músculo del lado derecho: es algo común a muchos canales.
  - Coincide con la prueba 3.
- Los valores son chicos (menos de 1 dB) porque Wiener usa un único `w` para toda la canción, y las ráfagas duran pocos segundos.

## §7. Conclusiones hasta ahora

**¿Obtuvimos un EEG limpio?**

- **En el caso armado (semi-sintético): sí, en buena medida.**
  - Con μ̃ = 0.03 y M = 3, `e` se parece a `s` mucho más que `y`: el RRMSE baja de 2.0 a 0.67 y la correlación es 0.78.
  - Además se cumple lo que dice la teoría: el límite de estabilidad μ̃ < 2, que `J(n)` llegue a `J_min`, y que el spread vuelva lento el aprendizaje.
- **En los datos reales de sub-01: no lo podemos afirmar todavía.**
  - Lo que sí funciona: el EMG de las ráfagas se va, y el ritmo α queda intacto.
  - Lo que no: (1) el ruido de ojos no desaparece, y a veces se resta de más; (2) las referencias de EMG también sacan alta frecuencia donde no hay músculo, que probablemente es EEG.

**Causas probables e ideas para probar:**

| Problema | Causa probable | Idea para probar |
|---|---|---|
| Los coeficientes de EMG aprenden lento | Spread de ~14 000 al apilar; NLMS normaliza por la potencia total | Normalizar cada referencia por separado |
| Se saca alta frecuencia sin ráfagas | Las referencias de EMG son electrodos de EEG y traen β/γ | Que el filtro de EMG solo aprenda en las ráfagas, o un pasa-altos más alto |
| Se resta de más el ruido de ojos | Un solo canal EOG para movimientos distintos | μ̃ más chico o VS-LMS; buscar datos con EOG vertical y horizontal |
| Hay poco ruido real para validar | El dataset viene preprocesado | Buscar sujetos con parpadeos, u otro dataset |